# Early Sepsis Onset Prediction

## End-to-End Pipeline Walkthrough

This notebook mirrors `main.py` and walks through every step of the sepsis prediction pipeline with inline explanations and plots.

**Task**: Predict sepsis onset **6 hours before** a physician's recorded diagnosis using ICU patient data from the PhysioNet Sepsis Challenge 2019 dataset.

**Key challenges addressed**:
- Label noise
- Temporal data leakage
- Irregular time-series with high missing-value rates
- Severe class imbalance

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# Ensure src/ is importable
sys.path.insert(0, os.path.abspath('..'))

RANDOM_STATE = 42
OUTPUT_DIR = '../outputs'
DATA_DIR = '../data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(RANDOM_STATE)
print('Setup complete.')

---
## Step 1: Load Data

Read all `.psv` patient files from `data/` and concatenate into a single DataFrame. Each file represents one ICU patient stay with hourly readings.

In [ ]:
from src.load_data import load_all_patients

df = load_all_patients(DATA_DIR)
df.head(10)

In [ ]:
# Quick look at missing-value rates
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 6))
missing_pct.head(20).plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('% Missing')
ax.set_title('Top 20 Columns by Missing Value Rate')
plt.tight_layout()
plt.show()

---
## Step 2: Label Engineering

Create the **6-hour-ahead prediction label** (`label_6h`):
- For each patient, find T = first hour where `SepsisLabel == 1`
- `label_6h = 1` for hours T-6 through T-1
- `label_6h = 0` for all hours before T-6
- **Drop** all rows at T and after (no post-diagnosis data)
- Non-sepsis patients: `label_6h = 0` everywhere

In [ ]:
from src.label_engineering import create_6h_labels, patient_level_split

df = create_6h_labels(df)

---
## Step 3: Patient-Level Train/Test Split

**Critical**: We split at the patient level so no patient appears in both sets. This prevents temporal data leakage.

In [ ]:
df_train, df_test = patient_level_split(df, test_size=0.2, random_state=RANDOM_STATE)

---
## Step 4: Feature Engineering

Compute **backward-looking** temporal features:
- Rolling mean, std, min, max (window=6 hours) for key vitals/labs
- Time-since-last-observation for sparse lab values
- ICULOS (ICU length of stay) as a direct feature

All rolling computations use `.shift(1)` to exclude the current hour.

In [ ]:
from src.features import engineer_features

df_train = engineer_features(df_train)
df_test = engineer_features(df_test)

---
## Step 5: Preprocessing

1. Add binary **missingness indicators** (before imputation)
2. **Forward-fill** per patient (simulate real-time data availability)
3. **Median imputation** using training-set medians
4. **StandardScaler** normalization (fit on train only)

⚠️ The scaler is fit **only** on training data and then applied to test data.

In [ ]:
from src.preprocessing import preprocess

df_train, df_test, scaler, feature_cols = preprocess(df_train, df_test, label_col='label_6h')
print(f'Feature columns: {len(feature_cols)}')

---
## Step 6: Label Noise Detection (cleanlab)

Use cleanlab's `CleanLearning` with a `RandomForestClassifier` to identify and remove likely mislabelled samples from the **training set only**.

In [ ]:
from src.label_noise import clean_training_data

df_train_clean, n_removed = clean_training_data(
    df_train, feature_cols, label_col='label_6h', random_state=RANDOM_STATE
)

# Prepare arrays
X_train = df_train_clean[feature_cols].values
y_train = df_train_clean['label_6h'].values.astype(int)
X_test = df_test[feature_cols].values
y_test = df_test['label_6h'].values.astype(int)

print(f'Final training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

---
## Step 7: Model Training & Calibration

### 7a. XGBoost (Baseline)
- `scale_pos_weight` handles class imbalance
- Early stopping on a 15% validation hold-out

In [ ]:
from src.model import train_xgboost, train_lightgbm
from src.calibration import calibrate_model, get_calibrated_predictions

xgb_model = train_xgboost(X_train, y_train, random_state=RANDOM_STATE, output_dir=OUTPUT_DIR)

In [ ]:
print('Calibrating XGBoost ...')
xgb_calibrated = calibrate_model(xgb_model, X_train, y_train)
xgb_probs = get_calibrated_predictions(xgb_calibrated, X_test)
print('Done.')

### 7b. LightGBM (Advanced)

In [ ]:
lgbm_model = train_lightgbm(X_train, y_train, random_state=RANDOM_STATE, output_dir=OUTPUT_DIR)

In [ ]:
print('Calibrating LightGBM ...')
lgbm_calibrated = calibrate_model(lgbm_model, X_train, y_train)
lgbm_probs = get_calibrated_predictions(lgbm_calibrated, X_test)
print('Done.')

---
## Step 8: Evaluation

For both models we compute:
- **Confusion matrix** (saved as PNG)
- **Classification report** (precision, recall, F1)
- **AUROC** and **AUPRC** (AUPRC is more informative for imbalanced data)
- **Calibration curve** (both models on one plot)
- **Threshold analysis** at 0.3, 0.4, 0.5, 0.6

In [ ]:
from src.evaluate import evaluate_all

prob_dict = {
    'XGBoost': xgb_probs,
    'LightGBM': lgbm_probs,
}

results = evaluate_all(y_test, prob_dict, output_dir=OUTPUT_DIR)

### Confusion Matrices

In [ ]:
from IPython.display import Image, display

for name in ['xgboost', 'lightgbm']:
    path = os.path.join(OUTPUT_DIR, f'confusion_matrix_{name}.png')
    if os.path.exists(path):
        print(f'\n{name.upper()} Confusion Matrix:')
        display(Image(filename=path))

### Calibration Curve

In [ ]:
cal_path = os.path.join(OUTPUT_DIR, 'calibration_curve.png')
if os.path.exists(cal_path):
    display(Image(filename=cal_path))

---
## Summary

| Metric | XGBoost | LightGBM |
|--------|---------|----------|
| AUROC  | See above | See above |
| AUPRC  | See above | See above |

**Key takeaways**:
- AUPRC is the more informative metric for this highly imbalanced task.
- Calibration curves show whether predicted probabilities match observed frequencies.
- The threshold analysis helps clinicians choose an operating point that balances sensitivity and specificity.
- Label noise removal with cleanlab improved training data quality.

Results and plots are saved in the `outputs/` directory.